# Аудит LoRA v1: где он помогает и сколько данных не использовано

Fine-tuned USER-bge-m3 (stage 8A/8B) — самый весомый канал попытки 2
(вес 1.25). Перед его переобучением нужно понять две вещи:

1. **Где fine-tune даёт прирост.** Benchmark на 72% состоит из редких
   запросов (train frequency `<=1`), holdout — на 15%. Если прирост живёт
   только на head, holdout переоценивает канал для benchmark.
2. **Сколько данных v1 не видел.** 8A брал только пары с объявлениями из
   benchmark-корпуса и по одной паре на объявление.

Гипотеза «прирост — это запоминание текстов из обучающих пар» тоже
проверяется: holdout исключал сигнатуры запросов, но не тексты.

Сравнение каналов — dense-only (global/local RRF с весами notebooks 05/08),
на всём holdout: оба канала не обучались на holdout-сигнатурах, и здесь
ничего не выбирается, поэтому dev/test не разделяются. Для каждого сегмента —
paired bootstrap 95% CI и one-sided randomization test.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import sys
import urllib.request

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.metrics import recall_at_k
from avito_retrieval.statistics import paired_recall_test

QUERY_COLUMNS = ["search_query", "search_location_id", "search_is_delivery_search",
                 "search_infm_params_text", "search_category"]
PAIRS_PATH = ROOT / "artifacts/finetuned_train_kaggle/finetune_pairs.parquet"


def clean(value: object) -> str:
    if value is None or (isinstance(value, float) and value != value):
        return ""
    return " ".join(str(value).replace("ё", "е").lower().split())


def parse_rank(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and value != value) or not str(value):
        return []
    return str(value).split()

## 1. Обучающие пары v1

Файл лежит в публичном output kernel 8A; при отсутствии скачивается
анонимным endpoint Kaggle (как `scripts/fetch_public_kaggle_outputs.py`).

In [2]:
if not PAIRS_PATH.exists():
    PAIRS_PATH.parent.mkdir(parents=True, exist_ok=True)
    api = ("https://www.kaggle.com/api/v1/kernels/output?userName=m1r0tvorxc"
           "&kernelSlug=avito-user-bge-m3-domain-adaptation")
    with urllib.request.urlopen(api, timeout=60) as response:
        files = json.load(response)["files"]
    url = next(f.get("url") or f.get("urlNullable") for f in files
               if f["fileName"].endswith("finetune_pairs.parquet"))
    urllib.request.urlretrieve(url, PAIRS_PATH)
v1_pairs = pd.read_parquet(PAIRS_PATH)
v1_texts = set(v1_pairs.query_text)
print(f"v1 training pairs: {len(v1_pairs):,}")

v1 training pairs: 17,033


## 2. Fine-tuned против zero-shot по сегментам

In [3]:
manifest = pd.read_parquet(ROOT / "artifacts/validation/manifest.parquet").copy()
labels = pd.read_parquet(ROOT / "artifacts/validation/labels.parquet")
relevant = labels.groupby("eval_query_id").item_id.agg(set).to_dict()
manifest["query_text"] = [f"{clean(q)} {clean(f)}".strip()
                          for q, f in zip(manifest.search_query, manifest.search_infm_params_text)]
manifest["seen_in_v1_pairs"] = manifest.query_text.isin(v1_texts)
manifest["is_tail"] = manifest.query_frequency.le(1)

benchmark = pd.read_parquet(ROOT / "dataset/benchmark_queries.parquet")
benchmark_texts = [f"{clean(q)} {clean(f)}".strip()
                   for q, f in zip(benchmark.search_query, benchmark.search_infm_params_text)]
exposure = {
    "holdout_seen_share": float(manifest.seen_in_v1_pairs.mean()),
    "holdout_head_seen_share": float(manifest.loc[~manifest.is_tail, "seen_in_v1_pairs"].mean()),
    "holdout_tail_seen_share": float(manifest.loc[manifest.is_tail, "seen_in_v1_pairs"].mean()),
    "benchmark_seen_share": float(np.mean([text in v1_texts for text in benchmark_texts])),
}
display(exposure)

rankings = pd.read_parquet(ROOT / "artifacts/dense_kaggle/dense_rankings.parquet").merge(
    pd.read_parquet(ROOT / "artifacts/finetuned_dense_kaggle/finetuned_dense_rankings.parquet"),
    on=["query_key", "split"], validate="one_to_one")
rankings = rankings[rankings.split.eq("validation")]
zero, fine = {}, {}
for row in rankings.itertuples(index=False):
    zero[row.query_key] = reciprocal_rank_fusion(
        [parse_rank(row.dense_global), parse_rank(row.dense_local)], weights=[1.0, 1.15], rrf_k=60, top_k=250)
    fine[row.query_key] = reciprocal_rank_fusion(
        [parse_rank(row.finetuned_global), parse_rank(row.finetuned_local)], weights=[1.0, 1.5], rrf_k=60, top_k=250)

segments = {
    "all": manifest,
    "tail (freq<=1)": manifest[manifest.is_tail],
    "head (freq>1)": manifest[~manifest.is_tail],
    "text seen in v1 pairs": manifest[manifest.seen_in_v1_pairs],
    "text unseen": manifest[~manifest.seen_in_v1_pairs],
}
rows = []
for name, frame in segments.items():
    ids = sorted(frame.eval_query_id)
    for k in (50, 250):
        test = paired_recall_test({q: fine[q][:k] for q in ids}, {q: zero[q][:k] for q in ids},
                                  relevant, query_ids=ids, k=k, n_resamples=20_000, seed=42)
        rows.append({"segment": name, "n": len(ids), "k": k,
                     "zero": recall_at_k({q: zero[q] for q in ids}, {q: relevant[q] for q in ids}, k=k),
                     "fine": recall_at_k({q: fine[q] for q in ids}, {q: relevant[q] for q in ids}, k=k),
                     **asdict(test)})
segment_table = pd.DataFrame(rows)
display(segment_table.round(4))

{'holdout_seen_share': 0.4094616639477977,
 'holdout_head_seen_share': 0.4805941542884523,
 'holdout_tail_seen_share': 0.0027397260273972603,
 'benchmark_seen_share': 0.07259380097879282}

,segment,n,k,zero,fine,mean_delta,ci_low,ci_high,p_value_greater,n_queries
0,all,2452,50,0.7681,0.8072,0.0390,0.0271,0.0510,0.0000,2452
1,all,2452,250,0.9035,0.9190,0.0155,0.0064,0.0244,0.0004,2452
2,tail (freq<=1),365,50,0.7616,0.7726,0.0110,-0.0247,0.0466,0.3244,365
3,tail (freq<=1),365,250,0.8986,0.9068,0.0082,-0.0164,0.0329,0.3435,365
4,head (freq>1),2087,50,0.7693,0.8132,0.0439,0.0312,0.0567,0.0000,2087
5,head (freq>1),2087,250,0.9044,0.9211,0.0167,0.0072,0.0263,0.0004,2087
6,text seen in v1 pairs,1004,50,0.7794,0.8141,0.0347,0.0191,0.0505,0.0001,1004
7,text seen in v1 pairs,1004,250,0.9057,0.9186,0.0129,-0.0002,0.0266,0.0295,1004
8,text unseen,1448,50,0.7603,0.8024,0.0420,0.0251,0.0593,0.0000,1448
9,text unseen,1448,250,0.9020,0.9193,0.0173,0.0052,0.0297,0.0031,1448


**Вывод.** Прирост одинаков для текстов, встречавшихся и не встречавшихся в
обучающих парах, то есть это не запоминание. Он сосредоточен на head:
на хвосте прирост статистически не отличается от нуля. Канал учит частые
типы запросов, а benchmark в основном из редких.

## 3. Сколько пар доступно для обучения

In [4]:
train = pd.read_parquet(ROOT / "dataset/train.parquet",
                        columns=QUERY_COLUMNS + ["item_id", "item_title_raw", "item_description_raw"])
marked = train.merge(manifest[QUERY_COLUMNS].drop_duplicates().assign(_holdout=1),
                     on=QUERY_COLUMNS, how="left")
usable = marked[marked._holdout.isna()].copy()
usable["query_text"] = [f"{clean(q)} {clean(f)}".strip()
                        for q, f in zip(usable.search_query, usable.search_infm_params_text)]
corpus_ids = set(pd.read_parquet(ROOT / "dataset/benchmark_items.parquet", columns=["item_id"]).item_id)
pairs = usable.drop_duplicates(["query_text", "item_id"])
per_text = pairs.groupby("query_text").size()
data_headroom = {
    "train_rows_after_holdout_exclusion": len(usable),
    "item_text_available_share": float(usable.item_title_raw.notna().mean()),
    "unique_query_item_pairs": len(pairs),
    "unique_query_texts": int(per_text.size),
    "unique_items": int(pairs.item_id.nunique()),
    "pairs_with_item_in_benchmark_corpus_share": float(pairs.item_id.isin(corpus_ids).mean()),
    "v1_pairs": len(v1_pairs),
    "v1_share_of_available": len(v1_pairs) / len(pairs),
    "pairs_kept_if_capped_at_32_per_text": int(per_text.clip(upper=32).sum()),
}
display(data_headroom)

(ROOT / "reports/finetune_audit_metrics.json").write_text(json.dumps({
    "exposure": exposure, "segments": segment_table.to_dict("records"),
    "data_headroom": data_headroom,
}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("saved reports/finetune_audit_metrics.json")

{'train_rows_after_holdout_exclusion': 491671,
 'item_text_available_share': 1.0,
 'unique_query_item_pairs': 457439,
 'unique_query_texts': 106817,
 'unique_items': 342481,
 'pairs_with_item_in_benchmark_corpus_share': 0.05802959520285765,
 'v1_pairs': 17033,
 'v1_share_of_available': 0.037235565835007506,
 'pairs_kept_if_capped_at_32_per_text': 304557}

saved reports/finetune_audit_metrics.json
